# 🫀 실험 1′ — Icentia11k 대규모 확증: **리듬 축은 스케일에서도 이기는가**

**MedKOS / `notebooks/exp1_icentia_rhythm_scale.ipynb`** · 퀘스트 `ailab-2026-0015`

---

## ⚠️ 먼저 — 왜 "실험 1"이 아니라 "실험 1′"인가

`mit-bih/PAPER.md` 를 확인한 결과, 원래 계획했던 **실험 1(보상성 휴지 → S 클래스 구제)은
이미 완료**되어 있습니다. 다시 할 필요가 없습니다.

| 원래 계획한 arm | 이미 한 것 | 결과 |
|---|---|---|
| A. 형태만 | `B2` CNN(raw beat) | 환자매크로 F1 **0.164** |
| B. + RR 정보 | `B3` CNN + RR | **0.484** — Δ **+0.320** [+0.246,+0.396], Bonferroni 통과 ★ |
| C. + 물리(보상지수) | `B4` 제안(RHYTHM innovation 포함) | 0.534 — B4−B3 Δ +0.050 **[−0.022,+0.122] 비유의** |

즉 **"리듬 정보를 넣어라"는 확증됐고**, **"보상지수를 별도 축으로 정교화하는 것"은
아직 판정이 안 났습니다.** 그리고 그 이유는 방법이 틀려서가 아니라 —

> `PAPER.md` §8.4 가설 3 · §10 Future Work 1:
> **SVDB 73환자로는 0.05 크기 효과의 검정력이 부족하다. 이를 배제하려면 수백 명 규모가 필요하다.**

**그래서 이 노트북이 하는 일은 새로운 방법을 만드는 게 아니라, 검정력을 올려서
이미 나온 "비유의" 판정을 확정 판정으로 바꾸는 것입니다.**

---

## 이 실험이 답하는 것

| | |
|---|---|
| **주 가설 (사전등록)** | 환자 수를 73 → 400+ 로 늘리면 `B3R − B3`(보상지수 축 추가)의 95% CI가 0을 벗어나는가 |
| **부 가설** | 형태 vs 형태+리듬 격차(Δ+0.320)가 **단일유도·다른 기기·다른 인구**에서도 재현되는가 |
| **왜 Icentia11k인가** | 11,000명 · 단일유도 250 Hz · **공개(신청 절차 없음)** · 검정력 문제의 유일한 해법 |
| **덤** | 여기서 만든 단일유도 파이프라인이 그대로 **lead-agnostic 프로젝트의 출발점**이 됨 |

**검정력 근거**: SVDB 73환자에서 관측 CI 반폭 ≈ 0.073. CI는 √n에 반비례하므로
400명 → ≈0.031, 900명 → ≈0.021. **목표 효과 0.05를 판정할 수 있게 됩니다.**

---

## 사전등록 (실행 전에 고정 — 결과 보고 바꾸지 않기)

```
주지표   : 환자단위 매크로 F1 (N/S/V)          ← micro 금지 (PAPER §8.3)
비교     : B3R − B3, 대응 부트스트랩 B=2000, 환자 단위 리샘플
판정     : 95% CI가 0을 포함하지 않으면 '확증', 포함하면 '기각(효과 없음)'
저울     : seed 3개 확률 평균 앙상블            ← 단일 seed는 ±0.15 요동 (PAPER §0)
금지     : 테스트 환자로 어떤 선택도 하지 않는다. 임계·에폭·특징 전부 학습셋에서만.
```

> **이 셀을 실행하기 전에 위 사전등록을 읽고, 결과가 어느 쪽으로 나와도 그대로 보고한다고
> 정해두세요.** CI가 0을 포함하면 "보상지수 축은 효과 없음"이 결론이고, 그건 실패가 아니라
> `PAPER.md` §8.4의 세 가설 중 하나를 배제한 결과입니다.

---

## 실행 방법
1. 상단 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**
2. 셀을 **위에서부터 순서대로** 실행
3. **CELL 2(스모크 테스트)에서 한 번 멈추고 출력을 확인**하세요. 여기서 라벨 심볼이
   예상과 다르면 CELL 3의 매핑을 고쳐야 합니다.

**예상 시간** — `QUICK=True`: 다운로드 5–10분 + 학습 10분. `QUICK=False`: 30–50분 + 60–90분.


In [ ]:
# CELL 1 — 설치 · 설정 · 시드
!pip -q install wfdb

import os, json, time, random, warnings, numpy as np
from collections import Counter, defaultdict
warnings.filterwarnings("ignore")

# ══════════════════ 여기만 바꾸면 됩니다 ══════════════════
QUICK        = True      # True: 빠른 1회전(약 20분) / False: 본 실험(약 2시간)
USE_DRIVE    = True      # Drive에 캐시 저장(재실행 시 다운로드 생략)
DRIVE_DIR    = "MedKOS/ailab/exp1_icentia"
# ════════════════════════════════════════════════════════

if QUICK:
    N_PATIENTS, MINUTES, N_SEEDS, EPOCHS = 120, 12, 2, 8
else:
    N_PATIENTS, MINUTES, N_SEEDS, EPOCHS = 600, 15, 3, 12

FS        = 250              # Icentia11k 샘플링 주파수
W_PRE     = 100              # R 기준 앞 0.40 s
W_POST    = 150              # R 기준 뒤 0.60 s
CLASSES   = ["N", "S", "V"]  # Q(분류불가)는 제외 — support가 지표를 왜곡
TEST_FRAC = 0.35             # 환자 단위 분리 비율
SEED0     = 20260731

random.seed(SEED0); np.random.seed(SEED0)

CACHE = "/content/exp1_cache"
os.makedirs(CACHE, exist_ok=True)
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        CACHE = f"/content/drive/MyDrive/{DRIVE_DIR}"
        os.makedirs(CACHE, exist_ok=True)
    except Exception as e:
        print("Drive 마운트 생략:", e)

print(f"설정: 환자 {N_PATIENTS}명 · 환자당 {MINUTES}분 · seed {N_SEEDS}개 · epoch {EPOCHS}")
print(f"캐시: {CACHE}")

### CELL 2 — 스모크 테스트 (★ 여기서 한 번 멈추세요)

Icentia11k는 환자당 약 70분짜리 세그먼트 50개로 쪼개져 있습니다(`p00001_s00` 형태).
경로 규칙과 **주석 심볼**을 먼저 눈으로 확인합니다.

`sampto=`로 앞부분만 읽으므로 전체를 받지 않습니다 — 환자 1명당 수 MB만 내려옵니다.

In [ ]:
# CELL 2 — 스모크 테스트: 경로 규칙 + 주석 심볼 확인 (환자 2명)
import wfdb

PN_DIR = "icentia11k-continuous-ecg/1.0"

def rec_path(pid, seg):
    """Icentia11k 경로 규칙: p00/p00001/p00001_s00"""
    return f"p{pid//1000:02d}/p{pid:05d}/p{pid:05d}_s{seg:02d}"

def try_read(pid, seg=0, minutes=1):
    n = FS * 60 * minutes
    rec = wfdb.rdrecord(rec_path(pid, seg), pn_dir=PN_DIR, sampfrom=0, sampto=n)
    ann = wfdb.rdann(rec_path(pid, seg), "atr", pn_dir=PN_DIR, sampfrom=0, sampto=n)
    return rec, ann

ok = 0
for pid in (0, 1, 2, 3):
    try:
        rec, ann = try_read(pid)
        print(f"✅ p{pid:05d}  fs={rec.fs}  sig={rec.sig_name}  len={rec.p_signal.shape}")
        print("   주석 심볼 분포 :", Counter(ann.symbol).most_common())
        aux = [a for a in getattr(ann, "aux_note", []) if a and a.strip()]
        print("   리듬(aux_note) :", Counter(aux).most_common()[:6])
        ok += 1
        if ok >= 2:
            break
    except Exception as e:
        print(f"❌ p{pid:05d}: {type(e).__name__}: {str(e)[:120]}")

print("\n" + "="*72)
print("확인 사항")
print("  1) fs가 250인가")
print("  2) 심볼에 N(정상)·S(심방조기)·V(심실조기)·Q(분류불가)가 보이는가")
print("     → 다르면 아래 CELL 3의 SYMBOL_MAP 을 실제 심볼에 맞게 고칠 것")
print("  3) ❌만 나오면 경로 규칙이 다른 것 → PhysioNet 페이지의 파일 목록 확인")
print("="*72)

In [ ]:
# CELL 3 — 코호트 구축 (환자 단위 스트리밍 + 캐시)
# CELL 2 출력의 심볼이 다르면 여기를 고치세요.
SYMBOL_MAP = {
    "N": 0, "·": 0, "n": 0,          # 정상
    "S": 1, "A": 1, "a": 1, "J": 1,  # 심방 기원 조기박동 (SVEB)
    "V": 2, "E": 2,                  # 심실 기원 (VEB)
    "Q": 3, "?": 3,                  # 분류불가 → 학습·평가에서 제외
}

def robust_scale(x):
    med = np.median(x)
    iqr = np.percentile(x, 75) - np.percentile(x, 25)
    return ((x - med) / (iqr + 1e-6)).astype("float32")

def load_patient(pid, minutes=MINUTES, seg=0):
    """한 환자에서 비트 파형 + 라벨 + R시각(초)을 뽑는다. 실패하면 None."""
    n = FS * 60 * minutes
    p = rec_path(pid, seg)
    rec = wfdb.rdrecord(p, pn_dir=PN_DIR, sampfrom=0, sampto=n)
    ann = wfdb.rdann(p, "atr", pn_dir=PN_DIR, sampfrom=0, sampto=n)

    sig = robust_scale(rec.p_signal[:, 0].astype("float64"))
    keep = [i for i, s in enumerate(ann.symbol) if s in SYMBOL_MAP]
    if len(keep) < 60:
        return None
    samp = np.asarray(ann.sample)[keep]
    lab  = np.array([SYMBOL_MAP[ann.symbol[i]] for i in keep], dtype="int64")

    X, y, t = [], [], []
    for k in range(len(samp)):
        c = samp[k]
        if c - W_PRE < 0 or c + W_POST > len(sig):
            continue
        X.append(sig[c - W_PRE:c + W_POST])
        y.append(lab[k]); t.append(c / FS)
    if len(y) < 60:
        return None
    return np.array(X, "float16"), np.array(y), np.array(t, "float64")


CACHE_NPZ = os.path.join(CACHE, f"cohort_n{N_PATIENTS}_m{MINUTES}.npz")

if os.path.exists(CACHE_NPZ):
    d = np.load(CACHE_NPZ)
    Xb, yb, tb, pidb = d["X"], d["y"], d["t"], d["pid"]
    print(f"캐시 로드: {CACHE_NPZ}")
else:
    Xs, ys, ts, pids = [], [], [], []
    got, tried, t0 = 0, 0, time.time()
    while got < N_PATIENTS and tried < N_PATIENTS * 3:
        pid = tried; tried += 1
        try:
            out = load_patient(pid)
        except Exception:
            continue
        if out is None:
            continue
        X, y, t = out
        Xs.append(X); ys.append(y); ts.append(t)
        pids.append(np.full(len(y), pid, dtype="int32"))
        got += 1
        if got % 20 == 0:
            el = time.time() - t0
            print(f"  {got}/{N_PATIENTS}명  ({el:.0f}s, 예상 총 {el/got*N_PATIENTS/60:.1f}분)")
    Xb   = np.concatenate(Xs);  yb = np.concatenate(ys)
    tb   = np.concatenate(ts);  pidb = np.concatenate(pids)
    np.savez_compressed(CACHE_NPZ, X=Xb, y=yb, t=tb, pid=pidb)
    print(f"저장: {CACHE_NPZ}")

print(f"\n환자 {len(np.unique(pidb))}명 · 비트 {len(yb):,}개")
for c, name in enumerate(["N", "S", "V", "Q"]):
    m = yb == c
    print(f"  {name}: {m.sum():8,} ({m.mean():6.2%})  보유 환자 {len(np.unique(pidb[m])):4d}명")

### CELL 4 — 리듬 특징: 단순 RR vs RHYTHM innovation

두 종류를 만듭니다. **이 둘의 차이가 이 실험의 전부**입니다.

| | 내용 | 의미 |
|---|---|---|
| `RR` (B3용) | `rr_pre/rr_ref`, `rr_post/rr_ref`, `rr_ref` | **정보만** 준다 — 모델이 알아서 쓰라고 |
| `RHY` (B3R용) | RR 3개 + `pre_innov`, `post_innov`, **`comp = pre+post`**, `mad_ratio` | **법칙까지** 준다 — 생리학이 말하는 판별자를 직접 |

`comp`(보상지수)의 생리학 — `PAPER.md` §4.2:

* **PVC**: 심실 기원이라 역행 전도가 동방결절을 리셋하지 못함 → **완전 보상성 휴지** → `comp ≈ 0`
* **PAC**: 심방 기원이라 동방결절을 리셋함 → **불완전 보상성 휴지** → `comp < 0`

innovation은 환자별 인과적 EWMA 예측 잔차를 MAD로 정규화하고 tanh로 유계화한 것이라
**무차원**입니다 → 진폭·기기·샘플링에 불변 → 교차DB 전이가 원리적으로 가능.

In [ ]:
# CELL 4 — 리듬 특징 (PAPER.md §4.2 RHYTHM innovation 재구현)
# ⚠️ 이 셀은 CELL 3 직후 "한 번만" 실행하세요.
#    맨 아래에서 Q비트를 걸러내므로, 다시 돌리려면 CELL 3부터 다시 실행해야 합니다.
ALPHA = 0.3     # 인과적 EWMA 계수
K_REF = 10      # 국소 기준 RR 윈도

def rhythm_features(t, alpha=ALPHA):
    """한 환자의 R시각(초) → (RR특징[n,3], RHYTHM특징[n,7])."""
    n = len(t)
    rr = np.diff(t)                                  # rr[i] = t[i+1]-t[i]
    if len(rr) < 5:
        return None, None

    # ── 인과적 EWMA 예측(과거만 사용) ─────────────────────
    pred = np.empty_like(rr); acc = rr[0]
    for i in range(len(rr)):
        pred[i] = acc                                # 예측은 항상 '직전까지'
        acc = alpha * rr[i] + (1 - alpha) * acc
    resid = rr - pred

    mad_g = np.median(np.abs(resid - np.median(resid))) + 1e-9
    mad_l = np.empty_like(resid)
    for i in range(len(resid)):
        w = resid[max(0, i - 25):i + 26]
        mad_l[i] = np.median(np.abs(w - np.median(w)))
    scale = np.maximum(mad_l, 0.2 * mad_g) + 1e-9
    innov = np.tanh(resid / scale / 3.0)              # (-1,1) 유계

    rr_ref = np.array([np.median(rr[max(0, i-K_REF):i+K_REF+1]) for i in range(len(rr))])
    rr_ref = np.maximum(rr_ref, 1e-3)

    RR, RHY = [], []
    for k in range(n):
        i_pre, i_post = k - 1, k                      # rr_pre = rr[k-1], rr_post = rr[k]
        if i_pre < 0 or i_post >= len(rr):
            RR.append([1.0, 1.0, 0.8]); RHY.append([1.0, 1.0, 0.8, 0, 0, 0, 1.0]); continue
        ref = rr_ref[i_pre]
        rr_pre, rr_post = rr[i_pre], rr[i_post]
        pi, po = innov[i_pre], innov[i_post]
        RR.append([rr_pre/ref, rr_post/ref, ref])
        RHY.append([rr_pre/ref, rr_post/ref, ref,
                    pi, po, pi + po,                  # ★ comp = 보상지수
                    scale[i_pre] / mad_g])
    return np.array(RR, "float32"), np.array(RHY, "float32")


F_RR  = np.zeros((len(yb), 3), "float32")
F_RHY = np.zeros((len(yb), 7), "float32")
for pid in np.unique(pidb):
    m = pidb == pid
    a, b = rhythm_features(tb[m])
    if a is None:
        continue
    F_RR[m], F_RHY[m] = a, b

# Q(분류불가) 제거
keep = yb < 3
Xb, yb, tb, pidb = Xb[keep], yb[keep], tb[keep], pidb[keep]
F_RR, F_RHY = F_RR[keep], F_RHY[keep]
print(f"최종 비트 {len(yb):,}개 · 환자 {len(np.unique(pidb))}명 · 특징 RR{F_RR.shape} RHY{F_RHY.shape}")

### CELL 5 — ★ 물리 사전점검 (학습 전에 반드시)

**모델을 돌리기 전에, 법칙이 이 데이터에서 실제로 성립하는지 먼저 봅니다.**
S와 V의 보상지수 분포가 겹쳐 있다면 학습을 아무리 해도 이 축은 정보를 못 줍니다 —
그걸 2분 만에 알 수 있습니다.

In [ ]:
# CELL 5 — 물리 사전점검: 보상지수가 S와 V를 실제로 가르는가
import matplotlib.pyplot as plt

comp  = F_RHY[:, 5]                       # pre_innov + post_innov
prem  = F_RR[:, 0]                        # rr_pre / rr_ref (조기성)
ect   = prem < 0.85                       # 조기박동만

print("=== 전체 ===")
for c in range(3):
    m = yb == c
    print(f"  {CLASSES[c]}: n={m.sum():7,}  조기성 median={np.median(prem[m]):.3f}"
          f"   보상지수 median={np.median(comp[m]):+.3f}")

print("\n=== 조기박동(prem<0.85)만 — 여기서 S와 V가 갈려야 함 ===")
for c in (1, 2):
    m = ect & (yb == c)
    if m.sum() > 10:
        q1, q3 = np.percentile(comp[m], [25, 75])
        print(f"  {CLASSES[c]}: n={m.sum():6,}  보상지수 median={np.median(comp[m]):+.3f}"
              f"  IQR=[{q1:+.3f},{q3:+.3f}]")

mS, mV = ect & (yb == 1), ect & (yb == 2)
if mS.sum() > 10 and mV.sum() > 10:
    # 효과 크기(Cohen's d) — 분리력의 크기
    d = (np.mean(comp[mV]) - np.mean(comp[mS])) / np.sqrt(
        (np.var(comp[mV]) + np.var(comp[mS])) / 2 + 1e-9)
    print(f"\n  ▶ 보상지수의 S↔V 분리 효과크기 Cohen's d = {d:+.3f}")
    print("     |d|>0.8 크다 / 0.5~0.8 중간 / <0.2 거의 없음")
    print("     ※ 여기서 |d|<0.2면 이 축은 정보가 없다는 뜻 → 학습해도 안 오릅니다")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for c, col in zip(range(3), ["#888", "#c0392b", "#1c4780"]):
    ax[0].hist(comp[yb == c], bins=60, range=(-2, 2), alpha=.55,
               density=True, label=CLASSES[c], color=col)
ax[0].set_title("보상지수 — 전체 비트"); ax[0].legend(); ax[0].set_xlabel("pre_innov + post_innov")
for c, col in zip((1, 2), ["#c0392b", "#1c4780"]):
    m = ect & (yb == c)
    if m.sum() > 10:
        ax[1].hist(comp[m], bins=50, range=(-2, 2), alpha=.6,
                   density=True, label=CLASSES[c], color=col)
ax[1].axvline(0, ls="--", c="k", lw=1)
ax[1].set_title("보상지수 — 조기박동만 (S vs V)"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# CELL 6 — 환자 단위 분할 + 모델 (3 arms)
import tensorflow as tf
from tensorflow.keras import layers, models

pats = np.unique(pidb)
rng  = np.random.RandomState(SEED0); rng.shuffle(pats)
n_te = int(len(pats) * TEST_FRAC)
te_p, tr_p = set(pats[:n_te].tolist()), set(pats[n_te:].tolist())
is_te = np.isin(pidb, list(te_p))
print(f"학습 환자 {len(tr_p)}명 / 테스트 환자 {len(te_p)}명  (환자 겹침 없음)")
for name, m in (("train", ~is_te), ("test", is_te)):
    print(f"  {name}: {m.sum():7,}비트  " +
          "  ".join(f"{CLASSES[c]}={int(((yb==c)&m).sum()):,}" for c in range(3)))

# ── 클래스 가중치: effective number (PAPER.md §4.4, 학습셋에서만 유도) ──
def auto_weights(y, beta=0.9999):
    w = {}
    for c in range(3):
        n = max((y == c).sum(), 1)
        w[c] = (1 - beta) / (1 - beta ** n)
    s = w[0]
    return {c: float(v / s) for c, v in w.items()}

CW = auto_weights(yb[~is_te])
print("클래스 가중치(학습셋 유도):", {CLASSES[c]: round(v, 2) for c, v in CW.items()})


def build(n_feat, seed):
    tf.keras.utils.set_random_seed(seed)
    sig_in = layers.Input((W_PRE + W_POST, 1), name="beat")
    x = sig_in
    for f, k in ((32, 7), (64, 5), (128, 3)):
        x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(2)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)

    if n_feat > 0:
        f_in = layers.Input((n_feat,), name="feat")
        f = layers.Dense(32, activation="relu")(f_in)
        f = layers.Dense(32, activation="relu")(f)
        x = layers.Concatenate()([x, f])
        ins = [sig_in, f_in]
    else:
        ins = [sig_in]
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(3, activation="softmax")(x)
    m = models.Model(ins, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy")
    return m


ARMS = {
    "B2  형태만":        None,
    "B3  형태+RR":       F_RR,
    "B3R 형태+RHYTHM":   F_RHY,
}
Xf = Xb.astype("float32")[..., None]
print("\nARM 구성:", list(ARMS))

In [ ]:
# CELL 7 — 학습 (arm × seed) + seed 앙상블
def feat_norm(F, tr_mask):
    mu, sd = F[tr_mask].mean(0), F[tr_mask].std(0) + 1e-6
    return ((F - mu) / sd).astype("float32")

probs = {}                       # arm -> [n_test, 3] (seed 평균)
t0 = time.time()
for arm, F in ARMS.items():
    acc = np.zeros((int(is_te.sum()), 3), "float64")
    for s in range(N_SEEDS):
        seed = SEED0 + s
        if F is None:
            xtr, xte = Xf[~is_te], Xf[is_te]
        else:
            Fn = feat_norm(F, ~is_te)
            xtr, xte = [Xf[~is_te], Fn[~is_te]], [Xf[is_te], Fn[is_te]]
        m = build(0 if F is None else F.shape[1], seed)
        m.fit(xtr, yb[~is_te], epochs=EPOCHS, batch_size=512,
              class_weight=CW, verbose=0)
        acc += m.predict(xte, batch_size=1024, verbose=0)
        print(f"  {arm}  seed{s+1}/{N_SEEDS} 완료  ({time.time()-t0:.0f}s)")
        tf.keras.backend.clear_session()
    probs[arm] = acc / N_SEEDS
print(f"\n총 {time.time()-t0:.0f}초")

### CELL 8 — 평가: 환자단위 매크로 F1 + 대응 부트스트랩

`PAPER.md` §5.1 프로토콜 그대로입니다.

* **환자단위 매크로** — 비트를 다 모아서 계산하는 micro는 S가 많은 소수 환자에게 지배됩니다
  (MIT-BIH DS2에서 #232 하나가 S의 75.2%). 환자별로 F1을 내고 환자 평균을 냅니다.
* **대응 부트스트랩** — 같은 환자 집합을 리샘플해 arm 간 **차이**의 분포를 구합니다.
  환자마다 난이도가 달라서, 대응(paired)이 아니면 차이가 환자 분산에 묻힙니다.

In [ ]:
# CELL 8 — 환자단위 매크로 F1 + 대응 부트스트랩 CI
from sklearn.metrics import f1_score, classification_report, confusion_matrix

y_te, pid_te = yb[is_te], pidb[is_te]
upa = np.unique(pid_te)

def per_patient_macro(pred):
    """환자별 매크로 F1 배열(그 환자에 존재하는 클래스만 평균)."""
    out = []
    for p in upa:
        m = pid_te == p
        present = np.unique(np.concatenate([y_te[m], pred[m]]))
        out.append(f1_score(y_te[m], pred[m], labels=present,
                            average="macro", zero_division=0))
    return np.array(out)

res = {}
for arm, pr in probs.items():
    pred = pr.argmax(1)
    pp = per_patient_macro(pred)
    res[arm] = {"pred": pred, "pp": pp, "macro": float(pp.mean()),
                "f1_per_class": f1_score(y_te, pred, average=None,
                                         labels=[0, 1, 2], zero_division=0)}

print("="*74)
print(f"{'ARM':<20}{'환자매크로 F1':>16}{'N':>9}{'S':>9}{'V':>9}")
print("="*74)
for arm, r in res.items():
    f = r["f1_per_class"]
    print(f"{arm:<20}{r['macro']:>16.4f}{f[0]:>9.3f}{f[1]:>9.3f}{f[2]:>9.3f}")
print("="*74)

def paired_boot(a, b, B=2000, seed=SEED0):
    """환자 단위 대응 부트스트랩: mean(a-b)의 95% CI."""
    rs = np.random.RandomState(seed); d = a - b
    idx = rs.randint(0, len(d), size=(B, len(d)))
    boot = d[idx].mean(1)
    return float(d.mean()), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))

names = list(res)
comparisons = [(names[1], names[0]), (names[2], names[1]), (names[2], names[0])]
print(f"\n{'비교':<34}{'Δ':>9}{'95% CI':>22}   판정")
print("-"*80)
boots = {}
for hi, lo in comparisons:
    d, l, u = paired_boot(res[hi]["pp"], res[lo]["pp"])
    sig = "유의 ★" if (l > 0 or u < 0) else "비유의"
    boots[f"{hi} - {lo}"] = {"delta": d, "ci_low": l, "ci_high": u, "significant": l > 0 or u < 0}
    print(f"{hi.split()[0]} − {lo.split()[0]:<26}{d:>+9.4f}   [{l:+.4f}, {u:+.4f}]   {sig}")
print("-"*80)

key = f"{names[2]} - {names[1]}"
kb = boots[key]
print(f"\n▶ 사전등록 주가설 (B3R − B3, 보상지수 축의 순수 기여)")
print(f"   Δ = {kb['delta']:+.4f}  95% CI [{kb['ci_low']:+.4f}, {kb['ci_high']:+.4f}]")
print(f"   CI 반폭 = {(kb['ci_high']-kb['ci_low'])/2:.4f}   (SVDB 73환자 때는 0.073)")
if kb["significant"]:
    print("   → 확증: 보상지수 축은 단순 RR 위에 유의한 정보를 더한다")
else:
    print("   → 기각: 검정력을 올렸는데도 효과 없음 = 단순 RR로 이미 포화")
    print("     (PAPER.md §8.4 가설1 '정보 포화'를 지지 — 이것도 확정적 결과입니다)")

print("\n[참고] 비트풀링 micro (지배 효과 확인용, 주지표 아님)")
for arm, r in res.items():
    print(f"  {arm:<20} micro-macroF1 = {f1_score(y_te, r['pred'], average='macro'):.4f}")

In [ ]:
# CELL 9 — result.json 저장 + repo 반영 명령
best = names[2]
result = {
    "week": 1,
    "task": "Icentia11k 대규모 확증 — 리듬/보상지수 축 (실험1′)",
    "split": "inter",
    "metric": "macro_f1",
    "value": round(res[best]["macro"], 4),
    "passed": bool(boots[f"{names[2]} - {names[1]}"]["significant"]),
    "date": time.strftime("%Y-%m-%d"),
    "n_patients": int(len(np.unique(pidb))),
    "n_test_patients": int(len(upa)),
    "n_beats": int(len(yb)),
    "arms": {a: {"macro_f1": round(r["macro"], 4),
                 "f1_N": round(float(r["f1_per_class"][0]), 4),
                 "f1_S": round(float(r["f1_per_class"][1]), 4),
                 "f1_V": round(float(r["f1_per_class"][2]), 4)} for a, r in res.items()},
    "bootstrap": {k: {kk: (round(vv, 4) if isinstance(vv, float) else vv)
                      for kk, vv in v.items()} for k, v in boots.items()},
    "config": {"quick": QUICK, "minutes": MINUTES, "seeds": N_SEEDS, "epochs": EPOCHS},
}
for p in ["/content/result.json", os.path.join(CACHE, "result.json")]:
    try:
        json.dump(result, open(p, "w"), ensure_ascii=False, indent=2)
        print("저장:", p)
    except Exception as e:
        print("저장 실패", p, e)

print(json.dumps(result, ensure_ascii=False, indent=2)[:1400])
kb = boots[f"{names[2]} - {names[1]}"]
print(f"""
────────────────────────────────────────────────────────────────
repo에 실행 로그로 박기 (수치는 LLM이 아니라 이 파일에서 나옵니다)

  python pipelines/ingest_run.py --results result.json \\
      --notebook notebooks/exp1_icentia_rhythm_scale.ipynb \\
      --quest ailab-2026-0015 --step "exp1p-icentia-scale" \\
      --note "B3R-B3 delta {kb['delta']:+.4f} CI [{kb['ci_low']:+.4f},{kb['ci_high']:+.4f}] n_pat={len(np.unique(pidb))}"
────────────────────────────────────────────────────────────────""")

---

## 결과를 어떻게 읽나

| B3R − B3 의 95% CI | 결론 | 다음 |
|---|---|---|
| **0을 포함하지 않고 +** | 보상지수 축은 실재한다. `PAPER.md` §8.4 가설 3(표본 한계)이 원인이었음 | 이 축을 Physics Layer의 첫 제약으로 승격. `ailab-2026-0015` 실험 2로 |
| **0을 포함 (반폭 < 0.03)** | 단순 RR로 이미 **정보 포화**(가설 1). 확정적 음성 결과 | 물리를 *특징*으로 더 짜내지 말고, **loss·구조 제약**으로 넣는 쪽으로 전환하거나 유도 확장(실험 9·10)으로 |
| **0을 포함 (반폭 > 0.05)** | 아직 검정력 부족 | `QUICK=False` + `N_PATIENTS`를 900 이상으로 올려 재실행 |

**세 경우 모두 진행 가능한 결과입니다.** 음성이어도 `PAPER.md`의 세 가설 중 하나를
배제하므로, 그 자체로 논문에 들어갈 문장이 됩니다.

## 이 노트북이 남기는 자산
1. **단일유도(250 Hz) 비트 파이프라인** — lead-agnostic 프로젝트의 입력단이 그대로 됨
2. **환자단위 매크로 + 대응 부트스트랩 평가 루틴** — 이후 모든 실험의 공용 저울
3. **물리 사전점검 셀** — 새 물리 축을 넣을 때마다 학습 전에 2분 만에 걸러내는 도구

## 하지 않은 것 (정직하게)
- Icentia11k 라벨은 기술자 1인 전수 판독이라 MIT-BIH의 2인 심장전문의 합의와 **품질 등급이 다릅니다**. 절대 성능을 MIT-BIH·SVDB 수치와 직접 비교하면 안 됩니다 — **arm 간 차이만** 비교하세요.
- 환자당 앞부분 N분만 사용합니다(다운로드 비용). 기록 후반의 리듬 변화는 안 봅니다.
- 임계값 최적화를 하지 않았습니다(argmax). 동작점 실험은 별도입니다.
